____
__Universidad Tecnologica Nacional Buenos Aires__<br/>
__Ciencia de Datos ClusterAI__<br/>
__Ejercicio: Regresion Lineal, Ridge y Lasso__<br/>
____

## Consigna

En este ejercicio vas a hacer un análisis de regresión sobre el dataset **Diabetes** de scikit-learn, comparando **Regresión Lineal (OLS)**, **Ridge** y **Lasso**.

El dataset tiene 442 pacientes y 10 features clínicas. La variable objetivo es una medida cuantitativa de la progresión de la enfermedad un año después de la medición inicial.

Completá las celdas marcadas con **`# TODO`**. Al final hay preguntas de reflexión.

#### Importamos librerías

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
sns.set_context("talk", font_scale=0.85)

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

#### Cargamos el dataset

In [ ]:
diabetes = load_diabetes()
x = diabetes.data
y = diabetes.target
feature_names = list(diabetes.feature_names)
print("Features:", feature_names)
print("Shape:", x.shape)

## Variables del dataset Diabetes

| Variable | Descripción |
|----------|-------------|
| **age** | Edad del paciente |
| **sex** | Sexo |
| **bmi** | Índice de masa corporal |
| **bp** | Presión arterial media |
| **s1** | Colesterol total en suero (tc) |
| **s2** | Lipoproteínas de baja densidad (ldl) |
| **s3** | Lipoproteínas de alta densidad (hdl) |
| **s4** | Colesterol total / HDL (tch) |
| **s5** | Log del nivel de triglicéridos en suero (ltg) |
| **s6** | Nivel de azúcar en sangre (glu) |
| **target** *(objetivo)* | Progresión de la enfermedad un año después de la medición inicial |

## 1. Exploración de datos

In [ ]:
df = pd.DataFrame(x, columns=feature_names)
df['target'] = y
df.head()

In [ ]:
df.describe()

In [ ]:
# Distribución de la variable objetivo
sns.histplot(y, bins=25)
plt.title('Distribución del target')
plt.show()

In [ ]:
# TODO: graficá la correlación de cada feature con el target, ordenada de menor a mayor
# Pista: correlaciones = df.corr()['target'].drop('target').sort_values()
#        sns.barplot(x=correlaciones.values, y=correlaciones.index)


In [ ]:
# Correlación entre pares de features (para detectar multicolinealidad)
sns.heatmap(df[feature_names].corr(), annot=True, fmt='.1f', annot_kws={'size': 7})
plt.show()
# TODO: ¿hay algún par de features muy correlacionadas entre sí? Anotá cuáles.

## 2. Train-test split y escalado

El scaler se fitea con el training y se aplica a train y test.

In [ ]:
xtr0, xte0, ytr, yte = train_test_split(x, y, test_size=0.3, random_state=10)

scaler = StandardScaler()
scaler.fit(xtr0)
xtr = scaler.transform(xtr0)
xte = scaler.transform(xte0)
print("Train:", xtr.shape, "Test:", xte.shape)

## 3. Train-validation split con KFold

Tomamos el primer fold para inspeccionar los pesos (más adelante hacemos cross validation completo).

In [ ]:
kf = KFold(n_splits=5)
train_index, val_index = next(kf.split(xtr))
xtra, xval = xtr[train_index], xtr[val_index]
ytra, yval = ytr[train_index], ytr[val_index]
print("Train:", xtra.shape, "Val:", xval.shape)

## 4. Regresión Lineal (OLS)

Usamos `LinearRegression` de scikit-learn.

In [ ]:
# TODO: creá el modelo LinearRegression() y entrenalo con .fit(xtra, ytra)
# ols = ...

# barplot de los pesos (ya armado)
sns.barplot(x=feature_names, y=ols.coef_)
plt.xticks(rotation=45, ha='right')
plt.title('Pesos OLS')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: predecí sobre validación con ols.predict(xval)
# y calculá el RMSE: np.sqrt(np.mean(np.square(yval - y_pred_ols)))
# rmse_ols = ...
# print('RMSE OLS:', rmse_ols)


## 5. Ridge Regression

Usamos `Ridge`, que agrega la penalización L2. El hiperparámetro `alpha` es el lambda de la teoría.

In [ ]:
lamb = 30

# TODO: creá Ridge(alpha=lamb) y entrenalo con .fit(xtra, ytra)
# ridge = ...

sns.barplot(x=feature_names, y=ridge.coef_)
plt.xticks(rotation=45, ha='right')
plt.title(f'Pesos Ridge (alpha={lamb})')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: compará la norma L2 de los pesos entre OLS y Ridge
# ¿Cuál es más chica? ¿Por qué?
# print('Norma OLS:  ', np.linalg.norm(ols.coef_))
# print('Norma Ridge:', np.linalg.norm(ridge.coef_))


In [ ]:
# TODO: predecí sobre validación con ridge.predict(xval) y calculá el RMSE


## 6. Regularization Path de Ridge

Vemos cómo cambian los pesos a medida que aumenta alpha.

In [ ]:
alphas = np.arange(1, 1000, 1)
weight_path = np.zeros((len(alphas), len(feature_names)))
rmse_path = np.zeros(len(alphas))

for i, a in enumerate(alphas):
    # TODO: entrená Ridge(alpha=a) con xtra, ytra
    #       guardá los pesos en weight_path[i, :] (usá .coef_)
    #       predecí sobre xval y guardá el RMSE en rmse_path[i]
    pass

In [ ]:
# Weight path: una línea por feature (ya armado)
for i, name in enumerate(feature_names):
    plt.plot(alphas, weight_path[:, i], label=name)
plt.xlabel('alpha')
plt.ylabel('Peso')
plt.title('Ridge Regression Path')
plt.legend(fontsize=7, loc='upper right')
plt.show()

In [ ]:
# TODO: graficá la curva de RMSE vs alpha (rmse_path contra alphas)


## 7. Lasso con Cross Validation

Ahora hacemos **cross validation completo**: iteramos sobre los 5 folds y promediamos el RMSE. Completá el cuerpo del doble loop.

In [ ]:
alphas_lasso = np.arange(0.1, 10, 0.1)
rmse_cv_lasso = np.zeros(len(alphas_lasso))

for i, a in enumerate(alphas_lasso):
    rmse_folds = []
    for train_idx, val_idx in kf.split(xtr):
        xtra_cv, xval_cv = xtr[train_idx], xtr[val_idx]
        ytra_cv, yval_cv = ytr[train_idx], ytr[val_idx]

        # TODO: entrená Lasso(alpha=a, max_iter=10000) con xtra_cv, ytra_cv
        #       predecí sobre xval_cv y agregá el RMSE a rmse_folds

    rmse_cv_lasso[i] = np.mean(rmse_folds)

idx_min = np.argmin(rmse_cv_lasso)
print(f'Alpha óptimo Lasso (CV): {alphas_lasso[idx_min]:.2f} — RMSE: {rmse_cv_lasso[idx_min]:.4f}')

In [ ]:
# TODO: graficá la curva de RMSE vs alpha del Lasso (rmse_cv_lasso contra alphas_lasso)


## 8. Evaluación final en test

Evaluamos los tres modelos en el test set, que no se usó en todo el análisis.

In [ ]:
# OLS
ols_final = LinearRegression().fit(xtr, ytr)
print('RMSE OLS en test:  ', np.sqrt(np.mean(np.square(yte - ols_final.predict(xte)))))

# Ridge
ridge_final = Ridge(alpha=30).fit(xtr, ytr)
print('RMSE Ridge en test:', np.sqrt(np.mean(np.square(yte - ridge_final.predict(xte)))))

# TODO: entrená Lasso con el alpha óptimo del CV sobre (xtr, ytr),
#       predecí en xte y mostrá el RMSE


## Preguntas de reflexión

Respondé en celdas de markdown:

1. ¿Por qué escalamos las features antes de entrenar? ¿Qué pasaría si no lo hiciéramos?
2. ¿Qué le pasa a la norma de los pesos a medida que aumenta alpha? Justificá con el regularization path.
3. ¿Qué diferencia notás entre Ridge y Lasso? ¿Qué hace Lasso con los pesos que Ridge no hace?
4. ¿Por qué usamos cross validation en vez de un solo fold de validación?
5. ¿Cuál de los tres modelos elegirías para este dataset y por qué?